# MinIO Data Migration Notebook

Migrate data from one MinIO bucket/connection to another interactively.

In [ ]:
import os
import sys
import logging
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional
import urllib3

try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(x, *args, **kwargs):
        return x
    print("Warning: tqdm not found. Install it with: pip install tqdm")

# Disable SSL warnings for self-signed certificates
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

try:
    from minio import Minio
    from minio.error import S3Error
except ImportError:
    print("Error: minio package not found. Install it with: pip install minio")

try:
    from dotenv import load_dotenv
except ImportError:
    print("Warning: python-dotenv not found. Install it with: pip install python-dotenv")
    load_dotenv = None

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True  # Force reconfiguration for Jupyter
)
logger = logging.getLogger(__name__)

In [ ]:
# Configuration

# Set these variables or use a .env file
ENV_FILE = ".env.migration"

def load_env_config(env_file: str) -> dict:
    """Load configuration from environment file."""
    config = {}
    
    if load_dotenv:
        load_dotenv(env_file)
    else:
        # Manual .env parsing
        if os.path.exists(env_file):
            with open(env_file, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith('#') and '=' in line:
                        key, value = line.split('=', 1)
                        os.environ[key.strip()] = value.strip().strip('"\'')
    
    # Source configuration
    config["source_endpoint"] = os.getenv("SOURCE_MINIO_ENDPOINT", "")
    config["source_access_key"] = os.getenv("SOURCE_MINIO_ACCESS_KEY", "")
    config["source_secret_key"] = os.getenv("SOURCE_MINIO_SECRET_KEY", "")
    config["source_bucket"] = os.getenv("SOURCE_MINIO_BUCKET", "")
    config["source_secure"] = os.getenv("SOURCE_MINIO_USE_SSL", "false").lower() == "true"
    
    # Destination configuration
    config["dest_endpoint"] = os.getenv("DEST_MINIO_ENDPOINT", "")
    config["dest_access_key"] = os.getenv("DEST_MINIO_ACCESS_KEY", "")
    config["dest_secret_key"] = os.getenv("DEST_MINIO_SECRET_KEY", "")
    config["dest_bucket"] = os.getenv("DEST_MINIO_BUCKET", "")
    config["dest_secure"] = os.getenv("DEST_MINIO_USE_SSL", "false").lower() == "true"
    
    return config

config = load_env_config(ENV_FILE)

# User configurable parameters (defaults from env, but you can override here)
SOURCE_ENDPOINT = config.get("source_endpoint") or "10.0.10.77:8621"
SOURCE_ACCESS_KEY = config.get("source_access_key") or "minioaccesskey"
SOURCE_SECRET_KEY = config.get("source_secret_key") or "miniosecretkey"
SOURCE_BUCKET = config.get("source_bucket") or "studio-production"
SOURCE_SECURE = config.get("source_secure", False)

DEST_ENDPOINT = config.get("dest_endpoint") or "ncc-dev-api-storage-data-product.qsncc.com"
DEST_ACCESS_KEY = config.get("dest_access_key") or "fitscan"
DEST_SECRET_KEY = config.get("dest_secret_key") or "hVQVu29W8pbaeNy"
DEST_BUCKET = config.get("dest_bucket") or "fitscan"
DEST_SECURE = config.get("dest_secure", False)

PREFIX = ""
WORKERS = 4
DRY_RUN = False
SKIP_EXISTING = True

print("Configuration Loaded:")
print(f"Source: {SOURCE_ENDPOINT} / {SOURCE_BUCKET} (SSL: {SOURCE_SECURE})")
print(f"Dest:   {DEST_ENDPOINT} / {DEST_BUCKET} (SSL: {DEST_SECURE})")

In [ ]:
class MinioMigrator:
    """Handles migration of objects between MinIO instances/buckets."""
    
    def __init__(
        self,
        source_endpoint: str,
        source_access_key: str,
        source_secret_key: str,
        source_bucket: str,
        source_secure: bool,
        dest_endpoint: str,
        dest_access_key: str,
        dest_secret_key: str,
        dest_bucket: str,
        dest_secure: bool,
        prefix: str = "",
        workers: int = 4,
        dry_run: bool = False,
        skip_existing: bool = True,
    ):
        self.source_bucket = source_bucket
        self.dest_bucket = dest_bucket
        self.prefix = prefix
        self.workers = workers
        self.dry_run = dry_run
        self.skip_existing = skip_existing
        
        # Initialize source client
        self.source_client = Minio(
            source_endpoint,
            access_key=source_access_key,
            secret_key=source_secret_key,
            secure=source_secure,
        )
        
        # Initialize destination client
        self.dest_client = Minio(
            dest_endpoint,
            access_key=dest_access_key,
            secret_key=dest_secret_key,
            secure=dest_secure,
        )
        
        # Statistics
        self.stats = {
            "total": 0,
            "migrated": 0,
            "skipped": 0,
            "failed": 0,
            "bytes_transferred": 0,
        }
    
    def validate_connections(self) -> bool:
        """Validate connections to both source and destination."""
        try:
            # Check source bucket exists
            if not self.source_client.bucket_exists(self.source_bucket):
                logger.error(f"Source bucket '{self.source_bucket}' does not exist")
                return False
            logger.info(f"✓ Connected to source bucket: {self.source_bucket}")
            
            # Check/create destination bucket
            if not self.dest_client.bucket_exists(self.dest_bucket):
                if self.dry_run:
                    logger.info(f"[DRY RUN] Would create destination bucket: {self.dest_bucket}")
                else:
                    logger.info(f"Creating destination bucket: {self.dest_bucket}")
                    self.dest_client.make_bucket(self.dest_bucket)
            logger.info(f"✓ Connected to destination bucket: {self.dest_bucket}")
            
            return True
        except S3Error as e:
            logger.error(f"Connection validation failed: {e}")
            return False
    
    def get_existing_objects(self) -> dict:
        """Get dict of existing objects in destination bucket with their size and etag."""
        existing = {}
        try:
            for obj in self.dest_client.list_objects(self.dest_bucket, prefix=self.prefix, recursive=True):
                existing[obj.object_name] = {
                    "size": obj.size,
                    "etag": obj.etag,
                }
        except S3Error as e:
            logger.warning(f"Could not list destination objects: {e}")
        return existing
    
    def migrate_object(self, obj_name: str, obj_size: int, obj_etag: str, existing_objects: dict) -> tuple:
        """Migrate a single object from source to destination (differential sync)."""
        try:
            # Check if object exists and compare for differential sync
            if self.skip_existing and obj_name in existing_objects:
                dest_obj = existing_objects[obj_name]
                # Compare size and etag - skip if identical
                if dest_obj["size"] == obj_size and dest_obj["etag"] == obj_etag:
                    logger.debug(f"Skipping unchanged object: {obj_name}")
                    return ("skipped", obj_name, 0)
                else:
                    logger.info(f"Object changed, re-syncing: {obj_name} (size: {dest_obj['size']} -> {obj_size})")
            
            if self.dry_run:
                logger.info(f"[DRY RUN] Would migrate: {obj_name} ({self._format_size(obj_size)})")
                return ("migrated", obj_name, obj_size)
            
            # Get object from source
            response = self.source_client.get_object(self.source_bucket, obj_name)
            
            # Get content type and metadata from source object stat
            stat = self.source_client.stat_object(self.source_bucket, obj_name)
            content_type = stat.content_type or "application/octet-stream"
            metadata = stat.metadata or {}
            
            # Upload to destination
            self.dest_client.put_object(
                self.dest_bucket,
                obj_name,
                response,
                length=obj_size,
                content_type=content_type,
                metadata=metadata,
            )
            
            response.close()
            response.release_conn()
            
            logger.info(f"✓ Migrated: {obj_name} ({self._format_size(obj_size)})")
            return ("migrated", obj_name, obj_size)
            
        except S3Error as e:
            logger.error(f"✗ Failed to migrate {obj_name}: {e}")
            return ("failed", obj_name, 0)
        except Exception as e:
            logger.error(f"✗ Unexpected error migrating {obj_name}: {e}")
            return ("failed", obj_name, 0)
    
    def migrate(self) -> dict:
        """Execute the migration."""
        logger.info("=" * 60)
        logger.info("Starting MinIO Migration")
        logger.info("=" * 60)
        logger.info(f"Source bucket: {self.source_bucket}")
        logger.info(f"Destination bucket: {self.dest_bucket}")
        logger.info(f"Prefix filter: {self.prefix or '(all objects)'}")
        logger.info(f"Workers: {self.workers}")
        logger.info(f"Skip existing: {self.skip_existing}")
        logger.info(f"Dry run: {self.dry_run}")
        logger.info("=" * 60)
        
        # Validate connections
        if not self.validate_connections():
            return self.stats
        
        # Get existing objects in destination (with size and etag for differential sync)
        existing_objects = {}
        if self.skip_existing:
            logger.info("Scanning destination for existing objects...")
            existing_objects = self.get_existing_objects()
            logger.info(f"Found {len(existing_objects)} existing objects in destination")
        
        # List all objects to migrate (including etag for comparison)
        logger.info("Scanning source bucket for objects...")
        objects_to_migrate = []
        for obj in self.source_client.list_objects(self.source_bucket, prefix=self.prefix, recursive=True):
            objects_to_migrate.append((obj.object_name, obj.size, obj.etag))
        
        self.stats["total"] = len(objects_to_migrate)
        logger.info(f"Found {self.stats['total']} objects to process")
        
        if not objects_to_migrate:
            logger.info("No objects to migrate")
            return self.stats
        
        # Calculate total size
        total_size = sum(obj[1] for obj in objects_to_migrate)
        logger.info(f"Total size: {self._format_size(total_size)}")
        logger.info("-" * 60)
        
        # Execute migration with thread pool
        start_time = datetime.now()
        
        with ThreadPoolExecutor(max_workers=self.workers) as executor:
            futures = {
                executor.submit(self.migrate_object, obj_name, obj_size, obj_etag, existing_objects): obj_name
                for obj_name, obj_size, obj_etag in objects_to_migrate
            }
            
            for future in tqdm(as_completed(futures), total=len(futures), desc="Migrating"):
                result, obj_name, size = future.result()
                if result == "migrated":
                    self.stats["migrated"] += 1
                    self.stats["bytes_transferred"] += size
                elif result == "skipped":
                    self.stats["skipped"] += 1
                elif result == "failed":
                    self.stats["failed"] += 1
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        # Print summary
        logger.info("=" * 60)
        logger.info("Migration Complete!")
        logger.info("=" * 60)
        logger.info(f"Total objects:     {self.stats['total']}")
        logger.info(f"Migrated:          {self.stats['migrated']}")
        logger.info(f"Skipped:           {self.stats['skipped']}")
        logger.info(f"Failed:            {self.stats['failed']}")
        logger.info(f"Data transferred:  {self._format_size(self.stats['bytes_transferred'])}")
        logger.info(f"Duration:          {duration:.2f} seconds")
        if duration > 0:
            speed = self.stats['bytes_transferred'] / duration
            logger.info(f"Speed:             {self._format_size(speed)}/s")
        logger.info("=" * 60)
        
        return self.stats
    
    @staticmethod
    def _format_size(size: int) -> str:
        """Format byte size to human readable string."""
        for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
            if size < 1024:
                return f"{size:.2f} {unit}"
            size /= 1024
        return f"{size:.2f} PB"

In [12]:
# Execute Migration

if not SOURCE_ENDPOINT or not DEST_ENDPOINT:
    logger.error("Missing Source or Destination endpoint configuration.")
    print("Please configure the source and destination in the configuration cell above.")
else:
    migrator = MinioMigrator(
        source_endpoint=SOURCE_ENDPOINT,
        source_access_key=SOURCE_ACCESS_KEY,
        source_secret_key=SOURCE_SECRET_KEY,
        source_bucket=SOURCE_BUCKET,
        source_secure=SOURCE_SECURE,
        dest_endpoint=DEST_ENDPOINT,
        dest_access_key=DEST_ACCESS_KEY,
        dest_secret_key=DEST_SECRET_KEY,
        dest_bucket=DEST_BUCKET,
        dest_secure=DEST_SECURE,
        prefix=PREFIX,
        workers=WORKERS,
        dry_run=DRY_RUN,
        skip_existing=SKIP_EXISTING,
    )
    
    stats = migrator.migrate()

2025-12-30 14:55:36 - INFO - ✓ Migrated: attachments/upload-queue/EF_SET_Certificate_1766379270375_2ac0bceb-e0a6-40b6-aaa3-09608fc77b70.pdf (449.26 KB)
2025-12-30 14:55:36 - INFO - ✓ Migrated: attachments/upload-queue/Eric_Kyaw_NOV_2025_CoverLetter_1763814217154_6d8526f7-268f-4daa-a783-d45bf2105655.txt (943.00 B)
2025-12-30 14:55:36 - INFO - ✓ Migrated: attachments/upload-queue/Emelyn_Bacani_Guest_Relations_Officer_FreshGraduate_are_welcome_CoverLetter_1761703959543_0df0ee40-5c3d-4cb4-b8a4-e69a6b71fa68.txt (776.00 B)
2025-12-30 14:55:36 - INFO - ✓ Migrated: attachments/upload-queue/Essential_Grammar__1759720223744_21a51cce-67cc-42b4-b20d-afd7f00b48e7.pdf (81.24 KB)
2025-12-30 14:55:36 - INFO - ✓ Migrated: attachments/upload-queue/Esther_Kim_NOV_2025_CoverLetter_1764310174952_1ea27e66-f510-4080-9d1b-3a970c9dc6a9.txt (1.47 KB)
2025-12-30 14:55:36 - INFO - ✓ Migrated: attachments/upload-queue/Fahrida_Klangjoho_AUG_2025_CoverLetter_1759220712665_f8e929a2-141b-4faf-b402-17129cf138cc.txt (1.